# CS3807 – Deep Learning Laboratory
# Experiment 5: Comprehensive Study of CNN Training, Regularization, Optimization,
# Hyperparameter Tuning, Transfer Learning and Cross-Validation

**Model:** MobileNetV2 &nbsp;|&nbsp; **Dataset:** Oxford-IIIT Pet Dataset &nbsp;|&nbsp; **Framework:** TensorFlow / Keras

This notebook follows the lab manual section-by-section. All plots are saved as separate PNG files at **600 DPI** in `/content/plots/`.
Inference cells are left blank (`_(write after running)_`) — to be filled in after execution, per instructions.

> Run cells top to bottom. Sections 5–10 use a **train/validation** split drawn from the Oxford Pet `train` split.
> The official `test` split is **never touched** until Section 12, per the manual's requirement.


## 0. Setup

In [ ]:
# Install / upgrade packages (Colab/Kaggle usually already have most of these).
# NOTE: we no longer install/import tensorflow_datasets - see the "Dataset loading" cell below
# for why, and how we load the Oxford-IIIT Pet dataset directly instead.
!pip install -q tensorflow scikit-learn matplotlib pandas seaborn pillow


In [ ]:
import os, time, json, random, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # a handful of files in this dataset are slightly truncated
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = 224
NUM_CLASSES = 37          # Oxford-IIIT Pet: 37 breeds
EPOCHS_SHORT = 6          # used for every comparison run (init / reg / BN / optimizer / hyperparam grid / transfer learning)
EPOCHS_CV = 6             # per fold in 5-fold CV
EPOCHS_FINAL = 15         # final retrained model only (runs once)
BATCH_SIZE_DEFAULT = 32

# Works whether this runs on Colab (/content) or Kaggle (/kaggle/working).
BASE_DIR = "/content" if os.path.exists("/content") else "/kaggle/working"
PLOT_DIR = os.path.join(BASE_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
print("TensorFlow version:", tf.__version__)


In [ ]:
def save_plot(fig, name):
    """Save a matplotlib figure at 600 DPI into PLOT_DIR and display it."""
    path = os.path.join(PLOT_DIR, f"{name}.png")
    fig.savefig(path, dpi=600, bbox_inches="tight")
    print(f"Saved: {path}")
    plt.show()


## 3. Dataset and Experimental Setup

Oxford-IIIT Pet Dataset (37 breeds, RGB, variable size). All images are resized to 224×224×3 and normalized using
MobileNetV2's `preprocess_input` (scales to [-1, 1]).

We download the dataset **directly from the official Oxford VGG mirror** (images.tar.gz + annotations.tar.gz) rather
than via `tensorflow_datasets`. `tensorflow_datasets` frequently breaks on Colab/Kaggle because whatever version of
`tensorflow_datasets` was preinstalled in the runtime often doesn't match the `tensorflow` version `pip install`
just pulled in — the import "succeeds" but the module is left in a broken half-initialized state, so `tfds.load`
appears to not exist. Downloading the official archives directly sidesteps that version-matching problem entirely
and works the same way on Colab or Kaggle.

We materialize the dataset into NumPy arrays (uint8, small enough to fit in memory: ~3,680 train images and ~3,669
test images) because Section 11 needs `sklearn`'s `StratifiedKFold` over the full training pool, which is easiest with
array indexing rather than a TFRecord pipeline.

- `X_pool`, `y_pool` — the official **trainval** split → used for (a) a train/val split in Sections 5–10, and (b) the
  5-fold CV pool in Section 11.
- `X_test`, `y_test` — the official **test** split → held untouched until Section 12.

> **Kaggle users:** open Notebook settings → Internet and turn it **On** before running this cell, or the download
> will fail.


In [ ]:
DATA_DIR = os.path.join(BASE_DIR, "oxford_pets_data")
os.makedirs(DATA_DIR, exist_ok=True)

IMAGES_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
ANNOTATIONS_URL = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"

def download_and_extract(url, dest_dir):
    fname = os.path.join(dest_dir, os.path.basename(url))
    if not os.path.exists(fname):
        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, fname)
    marker = fname + ".extracted"
    if not os.path.exists(marker):
        print(f"Extracting {fname} ...")
        with tarfile.open(fname) as tar:
            tar.extractall(dest_dir)
        open(marker, "w").close()

download_and_extract(IMAGES_URL, DATA_DIR)
download_and_extract(ANNOTATIONS_URL, DATA_DIR)

IMAGES_DIR = os.path.join(DATA_DIR, "images")
ANNOTATIONS_DIR = os.path.join(DATA_DIR, "annotations")

def read_annotation_list(txt_path):
    """Each line: <image_id> <class_id 1-37> <species 1|2> <breed_id>. We only need image_id + class_id."""
    entries = []
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            image_id, class_id = parts[0], int(parts[1])
            entries.append((image_id, class_id - 1))  # zero-index labels: 0-36
    return entries

def load_split_as_numpy(entries):
    images, labels = [], []
    skipped = 0
    for image_id, label in entries:
        path = os.path.join(IMAGES_DIR, image_id + ".jpg")
        try:
            img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        except Exception as e:
            skipped += 1
            continue
        images.append(np.array(img, dtype='uint8'))
        labels.append(label)
    if skipped:
        print(f"  (skipped {skipped} unreadable/corrupt image files out of {len(entries)})")
    return np.array(images, dtype='uint8'), np.array(labels, dtype='int32')

train_entries = read_annotation_list(os.path.join(ANNOTATIONS_DIR, "trainval.txt"))
test_entries = read_annotation_list(os.path.join(ANNOTATIONS_DIR, "test.txt"))

print("Loading train split (this can take a few minutes the first time - it downloads the dataset)...")
X_pool, y_pool = load_split_as_numpy(train_entries)
print("Loading test split...")
X_test, y_test = load_split_as_numpy(test_entries)

print("Pool (train) shape:", X_pool.shape, "| Test shape:", X_test.shape)
print("Number of classes seen:", len(np.unique(y_pool)))


In [ ]:
# Train / validation split from the pool (used for Sections 5-10).
# Stratified so every breed is represented in both train and val.
X_train, X_val, y_train, y_val = train_test_split(
    X_pool, y_pool, test_size=0.15, random_state=SEED, stratify=y_pool
)
print("Train:", X_train.shape, "Val:", X_val.shape, "Test (untouched):", X_test.shape)

def preprocess(x):
    x = tf.cast(x, tf.float32)
    return tf.keras.applications.mobilenet_v2.preprocess_input(x)

def make_dataset(X, y, batch_size=BATCH_SIZE_DEFAULT, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    def _map(img, label):
        img = preprocess(img)
        if augment:
            img = tf.image.random_flip_left_right(img)
        return img, label
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train, shuffle=True, augment=True)
val_ds   = make_dataset(X_val, y_val)
test_ds  = make_dataset(X_test, y_test)


## 4. MobileNetV2 Architecture

Key components: depthwise convolution, pointwise 1×1 convolution, inverted residual blocks, linear bottlenecks,
batch normalization, ReLU6 activation.


In [ ]:
base_preview = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), weights='imagenet', include_top=False)
print("Total layers in MobileNetV2 base:", len(base_preview.layers))
base_preview.summary()
del base_preview


## 5. Weight Initialization

To study initialization meaningfully we train MobileNetV2 **from scratch** (`weights=None`), re-initializing every
Conv2D / DepthwiseConv2D kernel with the strategy under test, and train each variant for a few epochs.

Strategies: **Zero**, **Random (Normal)**, **Xavier/Glorot**, **He**.


In [ ]:
INIT_MAP = {
    'zero':   tf.keras.initializers.Zeros(),
    'random': tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.05, seed=SEED),
    'xavier': tf.keras.initializers.GlorotUniform(seed=SEED),
    'he':     tf.keras.initializers.HeNormal(seed=SEED),
}

def build_model_with_initializer(init_name, num_classes=NUM_CLASSES):
    initializer = INIT_MAP[init_name]
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), weights=None, include_top=False)

    def clone_fn(layer):
        config = layer.get_config()
        # Regular/pointwise Conv2D layers use 'kernel_initializer'.
        if 'kernel_initializer' in config:
            config['kernel_initializer'] = tf.keras.initializers.serialize(initializer)
        # DepthwiseConv2D layers (the bulk of MobileNetV2) use 'depthwise_initializer'
        # instead - without this, most of the network stays at its default init
        # regardless of which strategy we think we're testing.
        if 'depthwise_initializer' in config:
            config['depthwise_initializer'] = tf.keras.initializers.serialize(initializer)
        return layer.__class__.from_config(config)

    base_cloned = tf.keras.models.clone_model(base, clone_function=clone_fn)
    x = tf.keras.layers.GlobalAveragePooling2D()(base_cloned.output)
    x = tf.keras.layers.Dense(128, activation='relu', kernel_initializer=initializer)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', kernel_initializer=initializer)(x)
    model = tf.keras.Model(base_cloned.input, outputs, name=f"mnv2_init_{init_name}")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
init_histories = {}
for name in INIT_MAP.keys():
    print(f"\n=== Training with {name} initialization ===")
    tf.keras.backend.clear_session()
    model = build_model_with_initializer(name)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_SHORT, verbose=1)
    init_histories[name] = hist.history
    del model


In [ ]:
# Plot 1: Training Loss vs Epoch (one curve per initialization method)
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in init_histories.items():
    ax.plot(range(1, EPOCHS_SHORT + 1), h['loss'], marker='o', label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Training Loss")
ax.set_title("Plot 1: Training Loss vs Epoch — Weight Initialization")
ax.legend(); ax.grid(alpha=0.3)
save_plot(fig, "plot01_training_loss_vs_epoch_initialization")


In [ ]:
# Plot 2: Validation Accuracy vs Epoch (one curve per initialization method)
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in init_histories.items():
    ax.plot(range(1, EPOCHS_SHORT + 1), np.array(h['val_accuracy']) * 100, marker='o', label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Plot 2: Validation Accuracy vs Epoch — Weight Initialization")
ax.legend(); ax.grid(alpha=0.3)
save_plot(fig, "plot02_val_accuracy_vs_epoch_initialization")


**Inference (Plots 1 & 2):** _(write after running)_

## 6. Regularization and Overfitting

From here on we use **ImageNet-pretrained** MobileNetV2 with the base **frozen** (feature extraction), and vary only
the classifier head: **No regularization**, **L2**, **Dropout**, **Batch Normalization**.


In [ ]:
def build_transfer_model(head='none', dropout_rate=0.5, l2_lambda=0.01,
                          optimizer=None, base_trainable=False,
                          fine_tune_at=None, num_classes=NUM_CLASSES):
    """General-purpose builder reused across Sections 6-10.
    head: 'none' | 'l2' | 'dropout' | 'batchnorm'
    """
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                              weights='imagenet', include_top=False)
    base.trainable = base_trainable
    if base_trainable and fine_tune_at is not None:
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)

    if head == 'l2':
        x = tf.keras.layers.Dense(128, activation='relu',
                                   kernel_regularizer=tf.keras.regularizers.l2(l2_lambda))(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax',
                                         kernel_regularizer=tf.keras.regularizers.l2(l2_lambda))(x)
    elif head == 'dropout':
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    elif head == 'batchnorm':
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    else:  # 'none'
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(base.input, outputs)
    model.base_model = base  # keep a direct handle to the base: Model(base.input, base.output) FLATTENS
                              # base's ~150 layers directly into model.layers rather than nesting it as
                              # one sub-model, so "isinstance(layer, tf.keras.Model)" cannot find it later.
                              # This handle is what Sections 10-12 use to unfreeze it for fine-tuning.
    if optimizer is None:
        optimizer = tf.keras.optimizers.Adam(1e-3)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
reg_configs = ['none', 'l2', 'dropout', 'batchnorm']
reg_histories = {}
for cfg in reg_configs:
    print(f"\n=== Training with regularization = {cfg} ===")
    tf.keras.backend.clear_session()
    model = build_transfer_model(head=cfg)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_SHORT, verbose=1)
    reg_histories[cfg] = hist.history
    del model


In [ ]:
# Plot 3: Training and Validation Accuracy vs Epoch (subplots, one per regularization config)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, cfg in zip(axes.flat, reg_configs):
    h = reg_histories[cfg]
    ax.plot(range(1, EPOCHS_SHORT + 1), np.array(h['accuracy']) * 100, marker='o', label='Train')
    ax.plot(range(1, EPOCHS_SHORT + 1), np.array(h['val_accuracy']) * 100, marker='s', label='Val')
    ax.set_title(cfg); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("Plot 3: Training vs Validation Accuracy — Regularization")
fig.tight_layout()
save_plot(fig, "plot03_train_val_accuracy_regularization")


In [ ]:
# Plot 4: Training and Validation Loss vs Epoch (subplots, one per regularization config)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, cfg in zip(axes.flat, reg_configs):
    h = reg_histories[cfg]
    ax.plot(range(1, EPOCHS_SHORT + 1), h['loss'], marker='o', label='Train')
    ax.plot(range(1, EPOCHS_SHORT + 1), h['val_loss'], marker='s', label='Val')
    ax.set_title(cfg); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("Plot 4: Training vs Validation Loss — Regularization")
fig.tight_layout()
save_plot(fig, "plot04_train_val_loss_regularization")


**Inference (Plots 3 & 4 — generalization gap):** _(write after running)_

## 7. Batch Normalization\n\nNumerical example from the manual, verified in code:

In [ ]:
x = np.array([2, 4, 6, 8], dtype=float)
mu = x.mean()
var = x.var()  # population variance (ddof=0), matches the manual's formula
std = np.sqrt(var)
x_hat = (x - mu) / std
print("mu_B     =", mu)
print("sigma^2_B =", var)
print("sqrt(sigma^2_B) =", std)
print("x_hat    =", np.round(x_hat, 3))
print("(gamma=1, beta=0) -> y =", np.round(x_hat, 3))


In [ ]:
# Plot 5: With vs Without Batch Normalization (validation accuracy vs epoch)
bn_histories = {
    'with_BN': reg_histories['batchnorm'],
    'without_BN': reg_histories['none'],
}
fig, ax = plt.subplots(figsize=(7, 5))
for name, h in bn_histories.items():
    ax.plot(range(1, EPOCHS_SHORT + 1), np.array(h['val_accuracy']) * 100, marker='o', label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Plot 5: With vs Without Batch Normalization")
ax.legend(); ax.grid(alpha=0.3)
save_plot(fig, "plot05_with_without_batchnorm")


**Inference (Plot 5):** _(write after running)_

## 8. Optimization Algorithms

Frozen pretrained base + a fixed classifier head (Dense(128, relu) → Dropout(0.5) → Dense(softmax)), trained with
**SGD**, **Momentum**, **RMSProp**, **Adam**.

**Controlled comparison:** all four optimizers use the **same learning rate (0.001)** so that the only factor being
varied is the optimization algorithm itself, consistent with the manual's "change one factor at a time" rule. Note
that vanilla SGD is commonly used with a larger LR (e.g. 0.01–0.1) in practice since it lacks per-parameter adaptive
scaling — holding LR fixed here is a deliberate simplification for a fair head-to-head comparison, and is worth
mentioning in the report if SGD/Momentum look artificially slow.


In [ ]:
OPTIMIZER_LR = 0.001  # held constant across all optimizers for a controlled comparison

def get_optimizer(name):
    if name == 'SGD':
        return tf.keras.optimizers.SGD(learning_rate=OPTIMIZER_LR)
    if name == 'Momentum':
        return tf.keras.optimizers.SGD(learning_rate=OPTIMIZER_LR, momentum=0.9)
    if name == 'RMSProp':
        return tf.keras.optimizers.RMSprop(learning_rate=OPTIMIZER_LR)
    if name == 'Adam':
        return tf.keras.optimizers.Adam(learning_rate=OPTIMIZER_LR)
    raise ValueError(name)

optimizer_names = ['SGD', 'Momentum', 'RMSProp', 'Adam']
opt_histories = {}
opt_times = {}
for name in optimizer_names:
    print(f"\n=== Training with optimizer = {name} ===")
    tf.keras.backend.clear_session()
    model = build_transfer_model(head='dropout', dropout_rate=0.5, optimizer=get_optimizer(name))
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_SHORT, verbose=1)
    opt_times[name] = time.time() - t0
    opt_histories[name] = hist.history
    del model


In [ ]:
# Plot 6: Training Loss vs Epoch for Different Optimizers
fig, ax = plt.subplots(figsize=(7, 5))
for name in optimizer_names:
    ax.plot(range(1, EPOCHS_SHORT + 1), opt_histories[name]['loss'], marker='o', label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Training Loss")
ax.set_title("Plot 6: Training Loss vs Epoch — Optimizers")
ax.legend(); ax.grid(alpha=0.3)
save_plot(fig, "plot06_training_loss_optimizers")


In [ ]:
# Plot 7: Validation Accuracy vs Epoch for Different Optimizers
fig, ax = plt.subplots(figsize=(7, 5))
for name in optimizer_names:
    ax.plot(range(1, EPOCHS_SHORT + 1), np.array(opt_histories[name]['val_accuracy']) * 100, marker='o', label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Plot 7: Validation Accuracy vs Epoch — Optimizers")
ax.legend(); ax.grid(alpha=0.3)
save_plot(fig, "plot07_val_accuracy_optimizers")


In [ ]:
# Optimizer summary table
rows = []
for name in optimizer_names:
    h = opt_histories[name]
    best_val_idx = int(np.argmax(h['val_accuracy']))
    rows.append({
        'Optimizer': name,
        'Final Loss': round(h['loss'][-1], 4),
        'Best Val. Accuracy (%)': round(max(h['val_accuracy']) * 100, 2),
        'Epoch to Converge (best val epoch)': best_val_idx + 1,
        'Time (s)': round(opt_times[name], 1),
    })
optimizer_table = pd.DataFrame(rows)
optimizer_table


**Inference (Plots 6 & 7, optimizer table):** _(write after running)_

## 9. CNN Hyperparameter Tuning

**Experimental rule:** change one hyperparameter at a time, keep everything else fixed at the baseline.

Baseline: optimizer = Adam, learning rate = 0.001, batch size = 32, dropout = 0.5 (frozen pretrained base).


In [ ]:
# Convolution output-size formula check (manual section 9): O = floor((N + 2P - K)/S) + 1
def conv_output_size(N, K, P, S):
    return (N + 2 * P - K) // S + 1

print("Example: N=224, K=3, P=1, S=2 ->", conv_output_size(224, 3, 1, 2))


In [ ]:
def train_with_config(lr=0.001, batch_size=BATCH_SIZE_DEFAULT, dropout_rate=0.5, epochs=EPOCHS_SHORT):
    tf.keras.backend.clear_session()
    tr_ds = make_dataset(X_train, y_train, batch_size=batch_size, shuffle=True, augment=True)
    va_ds = make_dataset(X_val, y_val, batch_size=batch_size)
    model = build_transfer_model(head='dropout', dropout_rate=dropout_rate,
                                  optimizer=tf.keras.optimizers.Adam(lr))
    hist = model.fit(tr_ds, validation_data=va_ds, epochs=epochs, verbose=0)
    del model
    return hist.history


In [ ]:
# --- Learning Rate sweep ---
lr_values = [0.001, 0.0001]
lr_results = {}
for lr in lr_values:
    print(f"Training with learning_rate={lr}")
    lr_results[lr] = train_with_config(lr=lr)

fig, ax = plt.subplots(figsize=(6, 5))
final_val_acc = [max(lr_results[lr]['val_accuracy']) * 100 for lr in lr_values]
ax.plot([str(v) for v in lr_values], final_val_acc, marker='o')
ax.set_xlabel("Learning Rate"); ax.set_ylabel("Best Validation Accuracy (%)")
ax.set_title("Plot 8: Learning Rate vs Validation Accuracy")
ax.grid(alpha=0.3)
save_plot(fig, "plot08_lr_vs_val_accuracy")


In [ ]:
# --- Batch Size sweep ---
batch_values = [16, 32, 64]
batch_results = {}
for bs in batch_values:
    print(f"Training with batch_size={bs}")
    batch_results[bs] = train_with_config(batch_size=bs)

fig, ax = plt.subplots(figsize=(6, 5))
final_val_acc = [max(batch_results[bs]['val_accuracy']) * 100 for bs in batch_values]
ax.plot([str(v) for v in batch_values], final_val_acc, marker='o')
ax.set_xlabel("Batch Size"); ax.set_ylabel("Best Validation Accuracy (%)")
ax.set_title("Plot 9: Batch Size vs Validation Accuracy")
ax.grid(alpha=0.3)
save_plot(fig, "plot09_batchsize_vs_val_accuracy")


In [ ]:
# --- Dropout Rate sweep ---
dropout_values = [0.0, 0.25, 0.5]
dropout_results = {}
for dr in dropout_values:
    print(f"Training with dropout_rate={dr}")
    dropout_results[dr] = train_with_config(dropout_rate=dr)

fig, ax = plt.subplots(figsize=(6, 5))
final_val_acc = [max(dropout_results[dr]['val_accuracy']) * 100 for dr in dropout_values]
ax.plot([str(v) for v in dropout_values], final_val_acc, marker='o')
ax.set_xlabel("Dropout Rate"); ax.set_ylabel("Best Validation Accuracy (%)")
ax.set_title("Plot 10: Dropout Rate vs Validation Accuracy")
ax.grid(alpha=0.3)
save_plot(fig, "plot10_dropout_vs_val_accuracy")


**Inference (Plots 8, 9, 10):** _(write after running)_

## 10. Transfer Learning and Fine-Tuning

**Case A — Feature Extraction:** pretrained base frozen, only the new classifier head trains.

**Case B — Fine-Tuning:** train Case A first for a few epochs (warm-up), then unfreeze the top layers of the base
and continue training with a small learning rate.


In [ ]:
FT_EPOCHS_WARMUP = 3
FT_EPOCHS_FINE = max(EPOCHS_SHORT - FT_EPOCHS_WARMUP, 1)  # remaining epochs after unfreezing (>=1 as a safety floor)
FINE_TUNE_AT = 100  # freeze all base layers before this index, unfreeze layers from this index onward

# --- Case A: Feature Extraction ---
tf.keras.backend.clear_session()
model_fe = build_transfer_model(head='dropout', dropout_rate=0.5, base_trainable=False)
hist_fe = model_fe.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_SHORT, verbose=1)
feature_extraction_history = hist_fe.history
del model_fe


In [ ]:
# --- Case B: Fine-Tuning (warm-up frozen, then unfreeze top layers with a small LR) ---
tf.keras.backend.clear_session()
model_ft = build_transfer_model(head='dropout', dropout_rate=0.5, base_trainable=False)
hist_warmup = model_ft.fit(train_ds, validation_data=val_ds, epochs=FT_EPOCHS_WARMUP, verbose=1)

# Unfreeze the top layers of the base for fine-tuning.
# NOTE: model_ft.layers does NOT contain a nested MobileNetV2 sub-model - Model(base.input, outputs)
# flattens the base's layers directly into model_ft. We use the model.base_model handle set inside
# build_transfer_model() instead of searching for a nested tf.keras.Model (which would never be found).
model_ft.base_model.trainable = True
for l in model_ft.base_model.layers[:FINE_TUNE_AT]:
    l.trainable = False

model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_finetune = model_ft.fit(train_ds, validation_data=val_ds, epochs=FT_EPOCHS_FINE, verbose=1)

fine_tuning_history = {
    'accuracy':      hist_warmup.history['accuracy'] + hist_finetune.history['accuracy'],
    'val_accuracy':  hist_warmup.history['val_accuracy'] + hist_finetune.history['val_accuracy'],
    'loss':          hist_warmup.history['loss'] + hist_finetune.history['loss'],
    'val_loss':      hist_warmup.history['val_loss'] + hist_finetune.history['val_loss'],
}
del model_ft


In [ ]:
# Plot 11: Feature Extraction vs Fine-Tuning (validation accuracy)
fig, ax = plt.subplots(figsize=(7, 5))
epochs_range = range(1, EPOCHS_SHORT + 1)
ax.plot(epochs_range, np.array(feature_extraction_history['val_accuracy']) * 100, marker='o', label='Feature Extraction')
ax.plot(epochs_range, np.array(fine_tuning_history['val_accuracy']) * 100, marker='s', label='Fine-Tuning')
ax.axvline(FT_EPOCHS_WARMUP + 0.5, color='gray', linestyle='--', alpha=0.6, label='Unfreeze point')
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Plot 11: Feature Extraction vs Fine-Tuning")
ax.legend(); ax.grid(alpha=0.3)
save_plot(fig, "plot11_feature_extraction_vs_finetuning")


In [ ]:
# Plot 12: Training and Validation Loss before/after fine-tuning
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(epochs_range, fine_tuning_history['loss'], marker='o', label='Train Loss')
ax.plot(epochs_range, fine_tuning_history['val_loss'], marker='s', label='Val Loss')
ax.axvline(FT_EPOCHS_WARMUP + 0.5, color='gray', linestyle='--', alpha=0.6, label='Unfreeze point')
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Plot 12: Training/Validation Loss — Before vs After Fine-Tuning")
ax.legend(); ax.grid(alpha=0.3)
save_plot(fig, "plot12_loss_before_after_finetuning")


**Discussion — why fine-tuning uses a smaller learning rate:** _(write after running)_

**Inference (Plots 11 & 12):** _(write after running)_

## 11. K-Fold Cross-Validation

Select 3–4 promising configurations from the studies above and evaluate each with **5-fold stratified cross-validation**
on the training pool (`X_pool`, `y_pool` — the official `trainval` split). The independent test set (`X_test`, `y_test`) is
**not** used here.

`candidate_configs` is now built **programmatically from the results already collected** in Sections 6, 8 and 9
(`best_reg_name`, `best_optimizer_name`, `best_lr`, `best_batch`, `best_dropout`) rather than hardcoded — so it
reflects whatever actually won in your run. It also includes a **fine-tuned** candidate (`C4_fine_tuned`), since the
manual treats fine-tuning as part of the overall model-selection process, not something evaluated only in Section 10.

**On fold scoring:** each fold uses `EarlyStopping(monitor='val_accuracy', restore_best_weights=True)` with
`patience = epochs` (so it never actually cuts training short within the few epochs we use) purely so that, at the
end of `fit`, Keras restores the weights from the fold's best epoch. We then call `model.evaluate()` on those restored
weights to get the fold's accuracy — i.e. a real "best checkpoint" score, not just the peak value read off the
training curve (which can be a mild lookahead / cherry-picking bias).


In [ ]:
# Determine the winners from earlier sections instead of hardcoding them.
best_reg_name = max(reg_histories, key=lambda k: max(reg_histories[k]['val_accuracy']))
best_optimizer_name = max(opt_histories, key=lambda k: max(opt_histories[k]['val_accuracy']))
best_lr = max(lr_results, key=lambda lr: max(lr_results[lr]['val_accuracy']))
best_batch = max(batch_results, key=lambda b: max(batch_results[b]['val_accuracy']))
best_dropout = max(dropout_results, key=lambda d: max(dropout_results[d]['val_accuracy']))

print("Best regularization:", best_reg_name)
print("Best optimizer:", best_optimizer_name)
print("Best learning rate:", best_lr)
print("Best batch size:", best_batch)
print("Best dropout rate:", best_dropout)

# A reg head of 'none' isn't a distinct architecture from the baseline, so fall back to 'dropout'
# (a real regularizer) when building a config meant to showcase "best regularization".
best_reg_head = best_reg_name if best_reg_name != 'none' else 'dropout'

candidate_configs = {
    'C1_baseline':          dict(head='none', optimizer_name='Adam', lr=0.001,
                                  dropout_rate=0.5, batch_size=32, finetune=False),
    'C2_best_reg':          dict(head=best_reg_head, optimizer_name='Adam', lr=0.001,
                                  dropout_rate=0.5, batch_size=32, finetune=False),
    'C3_best_opt_hparams':  dict(head='dropout', optimizer_name=best_optimizer_name, lr=best_lr,
                                  dropout_rate=best_dropout, batch_size=best_batch, finetune=False),
    'C4_fine_tuned':        dict(head='dropout', optimizer_name='Adam', lr=1e-5,
                                  dropout_rate=best_dropout, batch_size=best_batch, finetune=True,
                                  fine_tune_at=FINE_TUNE_AT, warmup_epochs=FT_EPOCHS_WARMUP),
}
candidate_configs


In [ ]:
def build_from_config(cfg):
    opt = get_optimizer(cfg['optimizer_name']) if cfg['optimizer_name'] in ('SGD', 'Momentum', 'RMSProp') \
        else tf.keras.optimizers.Adam(cfg['lr'])
    return build_transfer_model(head=cfg['head'], dropout_rate=cfg['dropout_rate'], optimizer=opt)

def train_and_get_best_val_acc(model, tr_ds, va_ds, epochs):
    """Train with early-stopping-based best-checkpoint restoration, then evaluate the
    restored (best) weights explicitly rather than reading the peak off the history curve."""
    es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max',
                                           patience=max(epochs, 1), restore_best_weights=True)
    model.fit(tr_ds, validation_data=va_ds, epochs=epochs, verbose=0, callbacks=[es])
    val_loss, val_acc = model.evaluate(va_ds, verbose=0)
    return val_acc * 100

def train_fold_config(cfg, X_tr, y_tr, X_va, y_va, epochs):
    tr_ds = make_dataset(X_tr, y_tr, batch_size=cfg['batch_size'], shuffle=True, augment=True)
    va_ds = make_dataset(X_va, y_va, batch_size=cfg['batch_size'])
    if cfg.get('finetune', False):
        model = build_transfer_model(head=cfg['head'], dropout_rate=cfg['dropout_rate'], base_trainable=False)
        warm_epochs = min(cfg.get('warmup_epochs', 3), max(epochs - 1, 1))
        model.fit(tr_ds, validation_data=va_ds, epochs=warm_epochs, verbose=0)
        # Same fix as Section 10: use model.base_model, not an isinstance search (see note there).
        model.base_model.trainable = True
        for l in model.base_model.layers[:cfg.get('fine_tune_at', FINE_TUNE_AT)]:
            l.trainable = False
        model.compile(optimizer=tf.keras.optimizers.Adam(cfg['lr']),
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        remaining = max(epochs - warm_epochs, 1)
        val_acc = train_and_get_best_val_acc(model, tr_ds, va_ds, remaining)
    else:
        model = build_from_config(cfg)
        val_acc = train_and_get_best_val_acc(model, tr_ds, va_ds, epochs)
    del model
    return val_acc

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {name: [] for name in candidate_configs}

for name, cfg in candidate_configs.items():
    print(f"\n===== 5-Fold CV: {name} =====")
    fold_accuracies = []
    for fold_i, (tr_idx, va_idx) in enumerate(skf.split(X_pool, y_pool), start=1):
        print(f"  Fold {fold_i}/5 ...")
        tf.keras.backend.clear_session()
        fold_acc = train_fold_config(cfg, X_pool[tr_idx], y_pool[tr_idx], X_pool[va_idx], y_pool[va_idx], EPOCHS_CV)
        fold_accuracies.append(fold_acc)
    cv_results[name] = fold_accuracies
    print(f"  {name}: folds={np.round(fold_accuracies,2)} mean={np.mean(fold_accuracies):.2f} sd={np.std(fold_accuracies):.2f}")


In [ ]:
# Cross-validation results table
cv_table_rows = []
for name, accs in cv_results.items():
    row = {'Configuration': name}
    for i, a in enumerate(accs, start=1):
        row[f'F{i}'] = round(a, 2)
    row['Mean ± SD'] = f"{np.mean(accs):.2f} ± {np.std(accs):.2f}"
    cv_table_rows.append(row)
cv_table = pd.DataFrame(cv_table_rows)
cv_table


In [ ]:
# Plot 13: 5-Fold Cross-Validation Accuracy with SD error bars
fig, ax = plt.subplots(figsize=(7, 5))
names = list(cv_results.keys())
means = [np.mean(cv_results[n]) for n in names]
sds = [np.std(cv_results[n]) for n in names]
ax.bar(names, means, yerr=sds, capsize=6, color='#4C72B0', alpha=0.85)
ax.set_ylabel("Mean Validation Accuracy (%)")
ax.set_xlabel("Hyperparameter Configuration")
ax.set_title("Plot 13: 5-Fold Cross-Validation Accuracy (± SD)")
plt.xticks(rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)
save_plot(fig, "plot13_cv_accuracy_with_error_bars")


**Inference (Plot 13 — mean performance and variability):** _(write after running)_

## 12. Final Model Evaluation

The manual asks for the final choice to weigh **mean CV accuracy, SD, and computational cost** together — not just
whichever config has the top mean. The cell below **auto-selects by mean CV accuracy** as a starting point and prints
mean/SD/fold-time context for all candidates side by side; inspect that printout and, if a lower-mean but
lower-variance or cheaper config looks like the better real-world choice, override `best_config_name` manually in the
next cell before continuing.


In [ ]:
cv_summary_rows = []
for name, accs in cv_results.items():
    cv_summary_rows.append({
        'Configuration': name,
        'Mean CV Acc (%)': round(np.mean(accs), 2),
        'SD': round(np.std(accs), 2),
        'Fine-tuned?': candidate_configs[name].get('finetune', False),
    })
cv_summary_df = pd.DataFrame(cv_summary_rows).sort_values('Mean CV Acc (%)', ascending=False)
print(cv_summary_df.to_string(index=False))

auto_best_config_name = cv_summary_df.iloc[0]['Configuration']
print(f"\nAuto-selected by mean CV accuracy: {auto_best_config_name}")
print("Review SD and cost above before finalizing - override best_config_name manually in the next cell if needed.")


In [ ]:
best_config_name = auto_best_config_name  # <-- override manually here if SD/cost argue for a different config
best_cfg = candidate_configs[best_config_name]
print("Selected configuration:", best_config_name, best_cfg)


In [ ]:
tf.keras.backend.clear_session()

def train_final_model(cfg, X_tr, y_tr, epochs):
    full_ds = make_dataset(X_tr, y_tr, batch_size=cfg['batch_size'], shuffle=True, augment=True)
    if cfg.get('finetune', False):
        model = build_transfer_model(head=cfg['head'], dropout_rate=cfg['dropout_rate'], base_trainable=False)
        warm_epochs = min(cfg.get('warmup_epochs', 3), max(epochs - 1, 1))
        model.fit(full_ds, epochs=warm_epochs, verbose=1)
        model.base_model.trainable = True
        for l in model.base_model.layers[:cfg.get('fine_tune_at', FINE_TUNE_AT)]:
            l.trainable = False
        model.compile(optimizer=tf.keras.optimizers.Adam(cfg['lr']),
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        model.fit(full_ds, epochs=max(epochs - warm_epochs, 1), verbose=1)
    else:
        model = build_from_config(cfg)
        model.fit(full_ds, epochs=epochs, verbose=1)
    return model

t0 = time.time()
final_model = train_final_model(best_cfg, X_pool, y_pool, EPOCHS_FINAL)
final_training_time = time.time() - t0

num_params = final_model.count_params()
print(f"Training time: {final_training_time:.1f}s | Parameters: {num_params:,}")


In [ ]:
# Evaluate on the held-out test set (used for the first and only time here)
eval_test_ds = make_dataset(X_test, y_test, batch_size=32)
y_pred_probs = final_model.predict(eval_test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
test_recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
test_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

final_report = {
    'Mean CV Accuracy (%)': round(np.mean(cv_results[best_config_name]), 2),
    'CV Standard Deviation': round(np.std(cv_results[best_config_name]), 2),
    'Test Accuracy (%)': round(test_accuracy * 100, 2),
    'Precision (macro)': round(test_precision, 4),
    'Recall (macro)': round(test_recall, 4),
    'F1-score (macro)': round(test_f1, 4),
    'Training Time (s)': round(final_training_time, 1),
    'Number of Parameters': num_params,
}
pd.DataFrame([final_report])


In [ ]:
# Plot 14: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cm, cmap='Blues')
ax.set_xlabel("Predicted Class"); ax.set_ylabel("True Class")
ax.set_title("Plot 14: Confusion Matrix (Final Model on Test Set)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
save_plot(fig, "plot14_confusion_matrix")

# Most confused class pairs (off-diagonal)
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)
top_confusions = np.dstack(np.unravel_index(np.argsort(-cm_offdiag.ravel())[:10], cm_offdiag.shape))[0]
print("Top confused (true_class_idx, predicted_class_idx, count):")
for t, p in top_confusions:
    if cm_offdiag[t, p] > 0:
        print(f"  true={t} pred={p} count={cm_offdiag[t,p]}")


In [ ]:
# Optional Plot 15: Misclassified Images
misclassified_idx = np.where(y_pred != y_test)[0]
sample_idx = np.random.choice(misclassified_idx, size=min(9, len(misclassified_idx)), replace=False)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(X_test[idx])
    ax.set_title(f"True: {y_test[idx]}  Pred: {y_pred[idx]}", fontsize=9)
    ax.axis('off')
fig.suptitle("Plot 15 (Optional): Misclassified Images")
fig.tight_layout()
save_plot(fig, "plot15_misclassified_images")


**Inference (Plot 14 — best classified / most confused classes / possible visual reasons):** _(write after running)_

**Inference (Plot 15, optional):** _(write after running)_

## 13. Overall Results

In [ ]:
# Assemble the overall summary table from variables collected in the sections above.
best_init_name = max(init_histories, key=lambda k: max(init_histories[k]['val_accuracy']))
best_reg_name = max(reg_histories, key=lambda k: max(reg_histories[k]['val_accuracy']))
best_optimizer_name = max(opt_histories, key=lambda k: max(opt_histories[k]['val_accuracy']))

overall_rows = [
    {'Configuration': 'Baseline (no reg, Adam, frozen)', 'CV Accuracy': '-', 'SD': '-',
     'Test Accuracy': '-', 'Training Time': f"{opt_times['Adam']:.1f}s"},
    {'Configuration': f'Best Initialization ({best_init_name})', 'CV Accuracy': '-', 'SD': '-',
     'Test Accuracy': '-', 'Training Time': '-'},
    {'Configuration': f'Best Regularization ({best_reg_name})', 'CV Accuracy': '-', 'SD': '-',
     'Test Accuracy': '-', 'Training Time': '-'},
    {'Configuration': f'Best Optimizer ({best_optimizer_name})', 'CV Accuracy': '-', 'SD': '-',
     'Test Accuracy': '-', 'Training Time': f"{opt_times[best_optimizer_name]:.1f}s"},
    {'Configuration': f'Best Hyperparameters / CV winner ({best_config_name})',
     'CV Accuracy': final_report['Mean CV Accuracy (%)'], 'SD': final_report['CV Standard Deviation'],
     'Test Accuracy': final_report['Test Accuracy (%)'], 'Training Time': f"{final_report['Training Time (s)']}s"},
    {'Configuration': 'Fine-Tuned Model', 'CV Accuracy': '-', 'SD': '-',
     'Test Accuracy': f"{max(fine_tuning_history['val_accuracy'])*100:.2f}", 'Training Time': '-'},
]
overall_table = pd.DataFrame(overall_rows)
overall_table


## 14. Required Inference for Plots

For every major plot, write 2–3 lines covering: (1) what the plot shows, (2) what trend is observed, (3) why that trend
might occur. Fill these in after running the notebook and reviewing the saved plots in `/content/plots/`.

- **Plot 1 (Training Loss vs Epoch — Initialization):** _(write after running)_
- **Plot 2 (Validation Accuracy vs Epoch — Initialization):** _(write after running)_
- **Plot 3 (Train/Val Accuracy — Regularization):** _(write after running)_
- **Plot 4 (Train/Val Loss — Regularization):** _(write after running)_
- **Plot 5 (With vs Without BN):** _(write after running)_
- **Plot 6 (Training Loss — Optimizers):** _(write after running)_
- **Plot 7 (Validation Accuracy — Optimizers):** _(write after running)_
- **Plot 8 (Learning Rate vs Val Accuracy):** _(write after running)_
- **Plot 9 (Batch Size vs Val Accuracy):** _(write after running)_
- **Plot 10 (Dropout Rate vs Val Accuracy):** _(write after running)_
- **Plot 11 (Feature Extraction vs Fine-Tuning):** _(write after running)_
- **Plot 12 (Loss Before/After Fine-Tuning):** _(write after running)_
- **Plot 13 (5-Fold CV Accuracy):** _(write after running)_
- **Plot 14 (Confusion Matrix):** _(write after running)_
- **Plot 15 (Misclassified Images, optional):** _(write after running)_


## 15. Discussion Questions

_(Answer after running the notebook and reviewing all results.)_

1. What is the difference between model parameters and hyperparameters? — _(answer)_
2. Why is weight initialization important? — _(answer)_
3. Why can zero initialization be problematic for neural networks? — _(answer)_
4. Compare Xavier and He initialization. — _(answer)_
5. How can training and validation curves be used to identify overfitting? — _(answer)_
6. How does Dropout reduce overfitting? — _(answer)_
7. What is the purpose of Batch Normalization? — _(answer)_
8. Explain the numerical Batch Normalization example. — _(answer)_
9. What are the roles of γ and β? — _(answer)_
10. Compare SGD, Momentum, RMSProp and Adam. — _(answer)_
11. What happens when the learning rate is too large? — _(answer)_
12. What happens when the learning rate is too small? — _(answer)_
13. What is the effect of increasing batch size? — _(answer)_
14. Explain stride and padding. — _(answer)_
15. Why is MobileNetV2 computationally efficient? — _(answer)_
16. What is depthwise separable convolution? — _(answer)_
17. What is transfer learning? — _(answer)_
18. Differentiate feature extraction and fine-tuning. — _(answer)_
19. Why is a smaller learning rate generally used during fine-tuning? — _(answer)_
20. Why is K-Fold Cross-Validation useful for hyperparameter selection? — _(answer)_
21. Why must the test set remain untouched during tuning? — _(answer)_
22. Why should mean and standard deviation both be reported? — _(answer)_
23. Is the highest validation accuracy always sufficient to select a model? — _(answer)_


## 16. Additional Exercise

Select two **new** combinations of learning rate, dropout, batch size and fine-tuning strategy, evaluate each with
5-fold cross-validation, and compare against the selected configuration (`best_config_name`) from Section 12.


In [ ]:
# EDIT these two new candidate configurations, then re-run this cell.
additional_configs = {
    'New_A': dict(head='dropout', optimizer_name='Adam', lr=0.0005, dropout_rate=0.4, batch_size=32),
    'New_B': dict(head='dropout', optimizer_name='RMSProp', lr=0.0001, dropout_rate=0.3, batch_size=64),
}

# Uses the same train_fold_config (EarlyStopping + restore_best_weights + evaluate) as Section 11,
# so results are directly comparable to cv_results.
additional_cv_results = {name: [] for name in additional_configs}
for name, cfg in additional_configs.items():
    cfg.setdefault('finetune', False)
    print(f"\n===== 5-Fold CV: {name} =====")
    fold_accuracies = []
    for fold_i, (tr_idx, va_idx) in enumerate(skf.split(X_pool, y_pool), start=1):
        print(f"  Fold {fold_i}/5 ...")
        tf.keras.backend.clear_session()
        fold_acc = train_fold_config(cfg, X_pool[tr_idx], y_pool[tr_idx], X_pool[va_idx], y_pool[va_idx], EPOCHS_CV)
        fold_accuracies.append(fold_acc)
    additional_cv_results[name] = fold_accuracies
    print(f"  {name}: mean={np.mean(fold_accuracies):.2f} sd={np.std(fold_accuracies):.2f}")

comparison_rows = []
for name, accs in {**{best_config_name: cv_results[best_config_name]}, **additional_cv_results}.items():
    comparison_rows.append({'Configuration': name, 'Mean (%)': round(np.mean(accs), 2), 'SD': round(np.std(accs), 2)})
pd.DataFrame(comparison_rows)


**Justification — is a new configuration preferable?** Consider accuracy, standard deviation, computational cost
and final test performance. — _(write after running)_